In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

target_schema = StructType([
    StructField("rep",        StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("target",     DoubleType(),  True),
    StructField("start_date", StringType(),  True),
])

targets = [
    ("Alice", "North", 3500.0, "2024-01-01"),
    ("Bob",   "South", 4000.0, "2024-01-01"),
    ("Carol", "East",  3000.0, "2024-01-01"),
]

df_targets = spark.createDataFrame(targets, target_schema)
df_targets.write.format("delta").mode("overwrite").saveAsTable("rep_targets")

print("rep_targets table ready")
spark.read.table("sales_reps").show()
spark.read.table("rep_targets").show()

In [0]:
%sql
select rep, amount, sale_date
from sales_reps
where amount > (
  select avg(amount) from sales_reps
)
order by amount desc

In [0]:
%sql 
select rep, total_revenue
from(
  select rep, round(sum(amount), 2) as total_revenue
  from sales_reps
  group by rep
) rep_totals
where total_revenue > 3000
order by total_revenue desc

In [0]:
%sql
-- Single CTE same result as the subquery above much cleaner

with rep_totals as (
  select
    rep,
    round(sum(amount),2) as total_revenue,
    count(sale_id) as total_sales
  from sales_reps
  group by rep
)
select *
from rep_totals
where total_revenue > 3000
order by total_revenue desc

In [0]:
%sql
-- Multiple CTEs chained together

with rep_totals as (
  select
    rep,
    round(sum(amount), 2) as total_revenue,
    round(avg(amount), 2 )as avg_sale,
    count(sale_id) as total_sales
  from sales_reps
  group by rep
),
rep_with_targets as (
  select
    r.rep,
    r.total_revenue,
    r.avg_sale,
    r.total_sales,
    t.target,
    round(r.total_revenue - t.target, 2) as vs_target
  from rep_totals r
  join rep_targets t on r.rep = t.rep
),
final as (
  select
    *,
    case 
      when vs_target > 0 then 'above target'
      when vs_target < 0 then 'below target'
      else 'on target'
    end as performance
  from rep_with_targets
)
select * 
from final
order by total_revenue desc

In [0]:
%sql
CREATE TABLE IF NOT EXISTS org_chart (
    employee_id INT,
    name        STRING,
    manager_id  INT,
    level       INT
) USING DELTA;

INSERT INTO org_chart VALUES
    (1, 'CEO',      NULL, 1),
    (2, 'VP Sales', 1,    2),
    (3, 'VP Tech',  1,    2),
    (4, 'Alice',    2,    3),
    (5, 'Bob',      2,    3),
    (6, 'Carol',    3,    3);

In [0]:
%sql
-- Walk up the org chart from any employee to CEO
WITH RECURSIVE org_path AS (
    SELECT employee_id, name, manager_id, 1 AS depth
    FROM org_chart
    WHERE name = 'Alice'

    UNION ALL

    SELECT o.employee_id, o.name, o.manager_id, op.depth + 1
    FROM org_chart o
    JOIN org_path op ON o.employee_id = op.manager_id
)
SELECT * FROM org_path
ORDER BY depth DESC

In [0]:
%sql
explain
select rep, round(sum(amount), 2) as total
from sales_reps
where region = 'North'
group by rep
order by total desc

In [0]:
%sql
-- bad: filters after joining - processes all rows first
select s.rep, s.amount, t.target 
from sales_reps s
join rep_targets t on s.rep = t.rep
where s.region = t.region

In [0]:
%sql 
-- good: filter before joining - smaller dataset goin into join
select s.rep, s.amount, t.target 
from(
  select rep, amount from sales_reps where region = 'North'
) s
join rep_targets t on s.rep = t.rep

-- 1. Filter early - reduce data before joining or aggregation
where region = 'North' -- before the join

-- 2. Never select * in production
select rep, amoun  -- not select *

-- 3. use CTEs over nested subqueries
with clean as (...)   - not select * from (select * from....)

-- 4. Avoid functions on filter columns   - kills index use
where year(sale_date) = 2024                           -- slower
where sale_date between '2024-01-01' and '2024-12-31'  -- faster

-- 5. Optimize delta tables that get queried often
Optimize sales_rep

-- 6. Use explain to check the plan before running heavy queries
Explain select ...